# Smoke tests for `geolab-base`

For every package in `environment.yml` (conda + pip): try to import it and
exercise one minimal API call. CLI-only packages get a `which`/`--version`
check instead. A failure here means something installed but doesn't load,
which is usually a sign of an ABI mismatch or a missing system library.

Run all cells. The summary at the bottom lists pass/fail per package.

## Setup

In [ ]:
import importlib
import shutil
import subprocess
import sys

RESULTS = []


def py(modname, alias=None, smoke=None):
    """Import `modname` and optionally run `smoke(mod)` as a sanity check."""
    label = alias or modname
    try:
        mod = importlib.import_module(modname)
        if smoke is not None:
            smoke(mod)
        version = getattr(mod, '__version__', '')
        RESULTS.append((label, 'OK', str(version), ''))
    except Exception as exc:
        RESULTS.append((label, 'FAIL', '', f'{type(exc).__name__}: {exc}'))


def cli(cmd, version_flag='--version'):
    """Verify `cmd` is on $PATH and responds to a version flag."""
    path = shutil.which(cmd)
    if not path:
        RESULTS.append((cmd, 'FAIL', '', 'not on $PATH'))
        return
    try:
        r = subprocess.run([cmd, version_flag],
                           capture_output=True, text=True, timeout=10)
        line = (r.stdout or r.stderr).strip().splitlines()
        version = line[0] if line else 'on PATH'
        RESULTS.append((cmd, 'OK', version[:80], ''))
    except Exception as exc:
        RESULTS.append((cmd, 'OK', 'on PATH', f'{type(exc).__name__}'))


print(f'Python {sys.version}')
print(f'sys.prefix: {sys.prefix}')

## Cloud & storage

In [ ]:
cli('aws')
py('awswrangler')
py('boto3', smoke=lambda m: m.client('s3', region_name='us-east-1'))
py('obstore')

## Geospatial

In [ ]:
py('cartopy.crs', alias='cartopy',
   smoke=lambda m: m.PlateCarree())
py('contextily')
py('fiona', smoke=lambda m: m.supported_drivers)
py('folium',
   smoke=lambda m: m.Map(location=[0, 0], zoom_start=2))
py('osgeo.gdal', alias='gdal',
   smoke=lambda m: m.VersionInfo('RELEASE_NAME'))
py('pyproj', smoke=lambda m: m.CRS('EPSG:4326'))
py('shapely.geometry', alias='shapely',
   smoke=lambda m: m.Point(0, 0).buffer(1).area)

## Core scientific stack

In [ ]:
py('numpy', smoke=lambda m: int(m.array([1, 2, 3]).sum()))
py('scipy.stats', alias='scipy', smoke=lambda m: m.norm.cdf(0))
py('pandas',
   smoke=lambda m: m.DataFrame({'a': [1, 2]}).shape)
py('geopandas')
import matplotlib; matplotlib.use('Agg')
py('matplotlib',
   smoke=lambda m: m.figure.Figure())
py('xarray',
   smoke=lambda m: m.DataArray([1, 2, 3]).sum().item())
py('netCDF4', alias='netcdf4')
py('h5py')
py('pyarrow',
   smoke=lambda m: m.array([1, 2, 3]).to_pylist())
py('bottleneck',
   smoke=lambda m: m.nansum([1.0, 2.0, float('nan'), 3.0]))
py('flox')
py('xarrayutils')
py('dask.array', alias='dask',
   smoke=lambda m: m.ones(10, chunks=5).sum().compute())
py('distributed')
py('dask_gateway', alias='dask-gateway')
py('cvxpy', smoke=lambda m: m.Variable(name='x'))

## Geo / geoscience

In [ ]:
py('dascore')
cli('gmt', version_flag='--version')
py('obspy',
   smoke=lambda m: m.UTCDateTime('2020-01-01').timestamp)
py('obsplus')
py('pygmt')

## Utilities

In [ ]:
py('tqdm',
   smoke=lambda m: list(m.tqdm(range(3), disable=True)))
py('requests')
py('yaml', alias='pyyaml',
   smoke=lambda m: m.safe_load('a: 1'))
cli('gs', version_flag='--version')      # ghostscript
cli('ffmpeg', version_flag='-version')

## Dev tools

In [ ]:
cli('gh')
cli('gh-scoped-creds')
py('pytest')
cli('ruff')

## Jupyter stack & extensions

In [ ]:
py('jupyterhub')
py('jupyter_server')
py('jupyterlab')
py('ipykernel')
py('jupyter_resource_usage', alias='jupyter-resource-usage')
py('jupyter_ruff', alias='jupyter-ruff')
py('jupyter_server_proxy', alias='jupyter-server-proxy')
py('jupyterlab_git', alias='jupyterlab-git')
py('jupyterlab_myst', alias='jupyterlab-myst')
py('jupyterlab_code_formatter')
py('jupyterlab_pygments')
py('nbdime')

## pip packages & visualization

In [ ]:
# EarthScope --------------------------------------------------
py('earthscope_sdk', alias='earthscope-sdk')
cli('es')                       # earthscope-cli entry point
py('earthscopestraintools')

# Jupyter add-ons ---------------------------------------------
py('jupyterlab_jupyterbook_navigation')

# Visualization & data frames ---------------------------------
py('altair',
   smoke=lambda m: m.Chart())
py('plotly')
py('polars',
   smoke=lambda m: m.DataFrame({'a': [1, 2]}))
py('vegafusion')
py('vl_convert', alias='vl-convert-python')
py('ipympl')
py('panel')

## Summary

In [ ]:
import pandas as pd
from IPython.display import display

df = pd.DataFrame(RESULTS,
                  columns=['package', 'status', 'version', 'error'])

passed = int((df['status'] == 'OK').sum())
total = len(df)
failed = total - passed

print(f'Results: {passed}/{total} OK, {failed} failed')
if failed:
    print('\nFailures:')
    for _, row in df[df['status'] == 'FAIL'].iterrows():
        print(f"  {row['package']:35s}  {row['error']}")

df.style.map(
    lambda v: ('color: red; font-weight: bold' if v == 'FAIL'
               else 'color: green'),
    subset=['status']
)